In [10]:
class PuzzleState:
    def __init__(self, board, parent=None, move=None, depth=0):
        if isinstance(board[0], (list, tuple)):
            self.board = tuple(tuple(row) for row in board)
        else:
            self.board = tuple(board[i*3:(i+1)*3] for i in range(3))

        self.parent = parent
        self.move = move
        self.depth = depth
        self.blank_pos = self._find_blank()

    def _find_blank(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c
        return None

    def get_moves(self):
        moves = []
        r_blank, c_blank = self.blank_pos
        directions = {
            'Up': (-1, 0),
            'Down': (1, 0),
            'Left': (0, -1),
            'Right': (0, 1)
        }

        for move_name, (dr, dc) in directions.items():
            new_r, new_c = r_blank + dr, c_blank + dc

            if 0 <= new_r < 3 and 0 <= new_c < 3:
                new_board_list = [list(row) for row in self.board]
                new_board_list[r_blank][c_blank], new_board_list[new_r][new_c] = \
                    new_board_list[new_r][new_c], new_board_list[r_blank][c_blank]

                moves.append(PuzzleState(new_board_list, self, move_name, self.depth + 1))
        return moves

    def is_goal(self, goal_board):
        if isinstance(goal_board[0], (list, tuple)):
            goal_board_2d = tuple(tuple(row) for row in goal_board)
        else:
            goal_board_2d = tuple(goal_board[i*3:(i+1)*3] for i in range(3))
        return self.board == goal_board_2d

    def get_path(self):
        path = []
        current = self
        while current:
            path.append((current.board, current.move))
            current = current.parent
        return path[::-1]

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(self.board)

    def __str__(self):
        return '\n'.join([' '.join(map(str, row)) for row in self.board])

In [11]:

class Cutoff:
    def __eq__(self, other):
        return isinstance(other, Cutoff)

CUTOFF = Cutoff()

def dls(state, goal_board, limit):
    if state.is_goal(goal_board):
        return state

    if state.depth == limit:
        return CUTOFF

    cutoff_occurred = False
    for neighbor in state.get_moves():
        result = dls(neighbor, goal_board, limit)
        if result is CUTOFF:
            cutoff_occurred = True
        elif result is not None:
            return result

    return CUTOFF if cutoff_occurred else None

In [12]:
def iterative_deepening_search(initial_board, goal_board):
    for limit in range(100):
        initial_state = PuzzleState(initial_board)
        result = dls(initial_state, goal_board, limit)

        if result is not CUTOFF:
            return result

    return None

In [13]:

initial_board = (
    1, 2, 3,
    4, 0, 5,
    6, 7, 8
)

goal_board = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0
)

print("Initial State:")
print(PuzzleState(initial_board))
print("\nGoal State:")
print(PuzzleState(goal_board))

print("\nSolving...")
solution_state = iterative_deepening_search(initial_board, goal_board)

if solution_state:
    path = solution_state.get_path()
    print(f"\nSolution Found at depth: {solution_state.depth}")
    for i, (board, move) in enumerate(path):
        if move:
            print(f"Step {i}: Move {move}")
        else:
            print(f"Step {i}: Initial State")
        print('\n'.join([' '.join(map(str, row)) for row in board]))
        print("-" * 10)
else:
    print("No solution found within the specified depth limit.")


Initial State:
1 2 3
4 0 5
6 7 8

Goal State:
1 2 3
4 5 6
7 8 0

Solving...

Solution Found at depth: 14
Step 0: Initial State
1 2 3
4 0 5
6 7 8
----------
Step 1: Move Right
1 2 3
4 5 0
6 7 8
----------
Step 2: Move Down
1 2 3
4 5 8
6 7 0
----------
Step 3: Move Left
1 2 3
4 5 8
6 0 7
----------
Step 4: Move Left
1 2 3
4 5 8
0 6 7
----------
Step 5: Move Up
1 2 3
0 5 8
4 6 7
----------
Step 6: Move Right
1 2 3
5 0 8
4 6 7
----------
Step 7: Move Down
1 2 3
5 6 8
4 0 7
----------
Step 8: Move Right
1 2 3
5 6 8
4 7 0
----------
Step 9: Move Up
1 2 3
5 6 0
4 7 8
----------
Step 10: Move Left
1 2 3
5 0 6
4 7 8
----------
Step 11: Move Left
1 2 3
0 5 6
4 7 8
----------
Step 12: Move Down
1 2 3
4 5 6
0 7 8
----------
Step 13: Move Right
1 2 3
4 5 6
7 0 8
----------
Step 14: Move Right
1 2 3
4 5 6
7 8 0
----------
